# QRE-3 - Colab Setup Notebook

Bootstrap a Colab session for the Qur'an Recitation Evaluation project.

## Required manual step

Set the runtime to **T4 GPU** before running cells:
**Runtime > Change runtime type > Hardware accelerator: T4 GPU > Save**.
Cell 4 (the smoke test) will fail loudly if the runtime is CPU-only.

## What this notebook does

1. Clones `MahmoudAlmodalal/Quran_Recitation_model` into `/content/`.
2. Installs project requirements (skipping `torch` / `torchaudio`, which
   Colab pre-installs against its own CUDA driver).
3. Verifies `torch.cuda.is_available()`.
4. Mounts Google Drive at `/content/drive`, prepares persistent folders,
   and emits a final acceptance-criteria summary.

## Drive layout convention (used by later notebooks and scripts)

```
/content/drive/MyDrive/quran-recitation-eval/
  data/         # raw and processed datasets
  checkpoints/  # model weights
  logs/         # training run CSVs
```

## Private-repo authentication (optional)

If the GitHub repo is private, add a Colab Secret:
**Key icon (left sidebar) > Add new secret > Name `GITHUB_TOKEN`,
Value: a fine-grained PAT with read access on this repo > toggle
"Notebook access" on.**

If no token is set, the clone falls through to an unauthenticated HTTPS
clone, which works only if the repo is public.

Note: when `GITHUB_TOKEN` is set, the PAT is interpolated into the clone
URL and visible in process arg lists. Acceptable for personal Colab
notebooks; do not use this pattern in shared environments.

## If pip prompts you to restart the runtime

Colab ships numpy 2.x; this notebook downgrades to numpy 1.26.4 for ABI
compatibility with torch 2.2.x and librosa. If Colab shows a "Restart
runtime" prompt after Cell 3, accept it and re-run from Cell 2.

In [ ]:
"""Clone the project repo into /content, or update it if already present."""
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/MahmoudAlmodalal/Quran_Recitation_model.git"
REPO_DIR = Path("/content/Quran_Recitation_model")

try:
    from google.colab import userdata  # type: ignore[import-not-found]

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

clone_url = (
    REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", clone_url, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
print(f"Repo ready at: {REPO_DIR}")
subprocess.run(["git", "log", "--oneline", "-3"], check=True)

In [ ]:
"""Install project requirements, skipping torch and torchaudio.

Colab pre-installs torch built against its own CUDA driver; forcing
torch==2.2.2 would either rebuild wheels for several minutes or trigger a
CUDA ABI mismatch. transformers==4.40.* supports any torch >=2.0, so
Colab's shipped torch works fine.
"""
import subprocess
from pathlib import Path

SKIP = {"torch", "torchaudio"}
pkgs: list[str] = []
for line in Path("requirements.txt").read_text().splitlines():
    s = line.strip()
    if not s or s.startswith("#"):
        continue
    name = s.split("==")[0].split(">")[0].split("<")[0].strip().lower()
    if name in SKIP:
        continue
    pkgs.append(s)

print(f"Installing {len(pkgs)} packages (torch/torchaudio kept from Colab):")
for p in pkgs:
    print(f"  {p}")
subprocess.run(["pip", "install", "-q", *pkgs], check=True)
print("Install complete.")

In [ ]:
"""torch.cuda.is_available() smoke test (per QRE-3 implementation notes)."""
import torch

assert torch.cuda.is_available(), (
    "CUDA unavailable. Set Runtime > Change runtime type > T4 GPU and re-run."
)
print(f"torch:        {torch.__version__}  (CUDA {torch.version.cuda})")
print(f"GPU:          {torch.cuda.get_device_name(0)}")

In [ ]:
"""Mount Google Drive, prepare persistent folders, and emit the final
acceptance-criteria summary."""
from pathlib import Path

import librosa
import torch
import transformers
from google.colab import drive  # type: ignore[import-not-found]

drive.mount("/content/drive")

PROJECT_DRIVE = Path("/content/drive/MyDrive/quran-recitation-eval")
for sub in ("data", "checkpoints", "logs"):
    (PROJECT_DRIVE / sub).mkdir(parents=True, exist_ok=True)

REPO = Path("/content/Quran_Recitation_model")
assert torch.cuda.is_available(), "CUDA must be available."
assert (REPO / "requirements.txt").is_file(), f"Repo not found at {REPO}"
assert PROJECT_DRIVE.is_dir(), f"Drive not mounted at {PROJECT_DRIVE}"

print("All acceptance checks passed.")
print(f"  torch:        {torch.__version__}  (CUDA {torch.version.cuda})")
print(f"  GPU:          {torch.cuda.get_device_name(0)}")
print(f"  transformers: {transformers.__version__}")
print(f"  librosa:      {librosa.__version__}")
print(f"  repo:         {REPO}")
print(f"  drive:        {PROJECT_DRIVE}")